In [ ]:
%pip install --quiet --upgrade diffusers transformers accelerate mediapy

In [ ]:
import mediapy as media
import random
import sys
import torch

from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
    )

pipe = pipe.to("cuda")

In [ ]:
import time

In [ ]:
prompt = "a photo of Pikachu fine dining with a view to the Eiffel Tower"
seed = random.randint(0, sys.maxsize)

num_inference_steps = 4

images = pipe(
    prompt = prompt,
    guidance_scale = 0.0,
    num_inference_steps = num_inference_steps,
    generator = torch.Generator("cuda").manual_seed(seed),
    ).images

print(f"Prompt:\t{prompt}\nSeed:\t{seed}")
media.show_images(images)
images[0].save("output.jpg")

In [ ]:
def generate_image(prompt):
  runtime = int(time.time())
  seed = random.randint(0, sys.maxsize)

  num_inference_steps = 4

  images = pipe(
      prompt = prompt,
      guidance_scale = 0.0,
      num_inference_steps = num_inference_steps,
      generator = torch.Generator("cuda").manual_seed(seed),
      ).images

  print(f"Prompt:\t{prompt}\nSeed:\t{seed}")
  filename = "output_{}.jpg".format(runtime)
  images[0].save("output_{}.jpg".format(runtime))
  return images, filename

In [ ]:
prompt = 'mdjrny-v4 style a beautiful anime cyborg girl with yellow eyes wearing a cat hoodie, pretty detailed eyes, full body. City background. posture by j scott campbell, perfect shading, soft studio lighting, ultra-realistic, photorealistic, octane render, cinematic lighting, hdr, in-frame, 4k, 8k, edge lighting'

images, filename = generate_image(prompt)
media.show_images(images)

In [ ]:
filename

In [ ]:
base_prompt = "phantasmal iridescent portrait of awoman, detailed, colourful, psychedelic, unreal engine, octane render, blender effect"

prompt = "mdjrny-v4 style " + base_prompt

images, fn = generate_image(prompt)
media.show_images(images)

In [ ]:
base_prompt = "Full body view of a Samurai warrior in the year 2432, 4k upscale, with London background, raining"
prompt = "mdjrny-v4 style " + base_prompt

images, fn = generate_image(prompt)
media.show_images(images)

In [ ]:
base_prompt = "Close-up Portrait of a cyborg geisha, a glossy white, black, red, ivory porcelain face, mechanical features, cybernetic eyes, baroque, rococo, anodized titanium highly detailed mechanisms, gears, fiber, cogs, bulbs, wires, cables, 70mm, Canon EOS 6D Mark II, 4k, 35mm (FX, Full-Frame), f/2.5, extremely detailed, very high details, photorealistic, hi res, hdr, UHD, hyper-detailed, ultra-realistic, vibrant, centered, vivid colors, Wide angle, zoom out"
prompt = "mdjrny-v4 style " + base_prompt

images, fn = generate_image(prompt)
media.show_images(images)


In [ ]:
base_prompt="Sam Bankman-Fried contagion"
prompt = "mdjrny-v4 style " + base_prompt

images, fn = generate_image(prompt)
media.show_images(images)

In [ ]:
base_prompt = """blush,long hair,yellow eyes{{{{{{ond eyes}}}}}}large breasts{{dark skin}}side from,
wide view,two reg,cowboy shot,from above,full body,whole body,Please draw a picture in the genre tha
is best at, change seasons, wind autumn leaves,Beautiful  mountain{{{{{palette,Hair that gradually becomes one with the background}}}}}dynamic angle
"""
prompt = "mdjrny-v4 style " + base_prompt

images, fn = generate_image(prompt)
media.show_images(images)

In [ ]:
prompt = "Portrait of cute Hogwarts student studying, peaceful expression, quill in hand, ancient tomes, magic glowing in the background, van gogh and da vinci inspired art style, swirling brushstrokes, warm colors, intricate details, mystical atmosphere, charming and whimsical"

images, fn = generate_image(prompt)
media.show_images(images)

# Lits of top animation hits

https://en.wikipedia.org/wiki/List_of_highest-grossing_animated_films

In [ ]:
prompt = "toy story"

images, fn = generate_image(prompt)
media.show_images(images)

In [ ]:
prompt = "animated sponge"

images, fn = generate_image(prompt)
media.show_images(images)

In [ ]:
prompt = "Rango"

images, fn = generate_image(prompt)
media.show_images(images)

In [ ]:
prompt = "The Lion King"

images, fn = generate_image(prompt)
media.show_images(images)

In [ ]:
prompt = "The Incredibles"
images, fn = generate_image(prompt)
media.show_images(images)

In [ ]:
prompt = "The Super Mario Bros. Movie"
images, fn = generate_image(prompt)
media.show_images(images)

In [ ]:
model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg11', pretrained=True)
# or any of these variants
# model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg11_bn', pretrained=True)
# model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg13', pretrained=True)
# model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg13_bn', pretrained=True)
# model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg16', pretrained=True)
# model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg16_bn', pretrained=True)
# model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg19', pretrained=True)
# model = torch.hub.load('pytorch/vision:v0.10.0', 'vgg19_bn', pretrained=True)
model.eval()

In [ ]:
# sample execution (requires torchvision)
from PIL import Image
from IPython.display import Image as Image2
from torchvision import transforms

In [ ]:
# Download ImageNet labels
!wget https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt

In [ ]:
# Read the categories
with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

In [ ]:
def classify_image(filename, k=5):
  input_image = Image.open(filename)
  preprocess = transforms.Compose([
      transforms.Resize(256),
      transforms.CenterCrop(224),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
  ])
  input_tensor = preprocess(input_image)
  input_batch = input_tensor.unsqueeze(0) # create a mini-batch as expected by the model

  # move the input and model to GPU for speed if available
  if torch.cuda.is_available():
      input_batch = input_batch.to('cuda')
      model.to('cuda')

  with torch.no_grad():
      output = model(input_batch)
  # Tensor of shape 1000, with confidence scores over ImageNet's 1000 classes
  #print(output[0])
  # The output has unnormalized scores. To get probabilities, you can run a softmax on it.
  probabilities = torch.nn.functional.softmax(output[0], dim=0)
  # Show top categories per image
  top_probs, top_catids = torch.topk(probabilities, k)

  return top_probs, top_catids

In [ ]:
filename = "output_1703914134.jpg"

Image2(filename=filename)

In [ ]:
top_probs, top_catids = classify_image(filename)

for i in range(top_probs.size(0)):
    print(categories[top_catids[i]], top_probs[i].item())

In [ ]:
filename = "output_1703914044.jpg"

Image2(filename=filename)

In [ ]:
top_probs, top_catids = classify_image(filename)

for i in range(top_probs.size(0)):
    print(categories[top_catids[i]], top_probs[i].item())

In [ ]:
filename = "output_1703913986.jpg"

Image2(filename=filename)

In [ ]:
top_probs, top_catids = classify_image(filename)

for i in range(top_probs.size(0)):
    print(categories[top_catids[i]], top_probs[i].item())

In [ ]:
filename = "output_1703915371.jpg"

Image2(filename=filename)

In [ ]:
top_probs, top_catids = classify_image(filename)

for i in range(top_probs.size(0)):
    print(categories[top_catids[i]], top_probs[i].item())

In [ ]:
filename = "output_1703915244.jpg"

Image2(filename=filename)

In [ ]:
top_probs, top_catids = classify_image(filename)

for i in range(top_probs.size(0)):
    print(categories[top_catids[i]], top_probs[i].item())